In [4]:
%pip install --upgrade numpy pandas matplotlib
%pip install --upgrade git+https://github.com/alemartinello/dstapi

Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/alemartinello/dstapi to /private/var/folders/14/62r75ltd7d7304xsr5sf3x2w0000gn/T/pip-req-build-pyaz7aeo
  Running command git clone --filter=blob:none --quiet https://github.com/alemartinello/dstapi /private/var/folders/14/62r75ltd7d7304xsr5sf3x2w0000gn/T/pip-req-build-pyaz7aeo
  Resolved https://github.com/alemartinello/dstapi to commit d9eeb5a82cbc70b7d63b2ff44d92632fd77123a4
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [7]:
# ============================================================
# INEQUALITY IN DENMARK
# Robust version using Statistics Denmark's official API
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from io import StringIO


# ============================================================
# 0. HELPER FUNCTIONS
# ============================================================

API_BASE = "https://api.statbank.dk/v1"


def get_metadata(table):
    """
    Download metadata for a StatBank table.
    This tells us the actual variable codes and value codes.
    """
    url = f"{API_BASE}/tableinfo/{table}?lang=en"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    return response.json()


def show_variables(metadata):
    """
    Print variable codes and their labels.
    Useful for checking what a table actually contains.
    """
    print("\nVariables in table:")
    for var in metadata["variables"]:
        print(f"{var['id']}  ->  {var['text']}")


def find_variable(metadata, possible_words):
    """
    Find a variable by searching its English label.
    possible_words should be a list such as:
    ['municipality', 'area']
    """

    for var in metadata["variables"]:

        text = str(var["text"]).lower()

        for word in possible_words:
            if word.lower() in text:
                return var

    raise ValueError(
        f"Could not find variable containing: {possible_words}"
    )


def find_value(variable, possible_words):
    """
    Find a value code inside a variable by searching its label.
    """

    values = variable["values"]
    value_texts = variable["valueTexts"]

    for code, text in zip(values, value_texts):

        text_lower = str(text).lower()

        for word in possible_words:
            if word.lower() in text_lower:
                return code

    raise ValueError(
        f"Could not find value containing: {possible_words}"
    )


def download_statbank(table, variables):
    """
    Download data from StatBank as CSV.

    variables must look like:
    [
        {"code": "...", "values": ["..."]},
        ...
    ]
    """

    payload = {
        "table": table,
        "format": "CSV",
        "lang": "en",
        "valuePresentation": "Value",
        "timeOrder": "Ascending",
        "variables": variables
    }

    response = requests.post(
        f"{API_BASE}/data",
        json=payload,
        timeout=60
    )

    response.raise_for_status()

    # Statistics Denmark CSV is semicolon-separated
    df = pd.read_csv(
        StringIO(response.text),
        sep=";"
    )

    return df


# ============================================================
# 1. GINI COEFFICIENT FROM IFOR41
# ============================================================

print("\n==============================")
print("IFOR41 - GINI")
print("==============================")


# Download metadata
meta41 = get_metadata("IFOR41")

show_variables(meta41)


# ------------------------------------------------------------
# Find the variables automatically
# ------------------------------------------------------------

indicator41 = find_variable(
    meta41,
    ["indicator", "unit"]
)

municipality41 = find_variable(
    meta41,
    ["municipality", "area"]
)

time41 = find_variable(
    meta41,
    ["year", "time"]
)


print("\nDetected IFOR41 variables:")
print("Indicator:", indicator41["id"])
print("Municipality:", municipality41["id"])
print("Time:", time41["id"])


# ------------------------------------------------------------
# Find Gini and All Denmark automatically
# ------------------------------------------------------------

gini_code = find_value(
    indicator41,
    ["gini"]
)

denmark41_code = find_value(
    municipality41,
    ["all denmark"]
)


print("\nDetected values:")
print("Gini code:", gini_code)
print("All Denmark code:", denmark41_code)


# ------------------------------------------------------------
# Download IFOR41
# ------------------------------------------------------------

gini_raw = download_statbank(
    "IFOR41",
    [
        {
            "code": indicator41["id"],
            "values": [gini_code]
        },
        {
            "code": municipality41["id"],
            "values": [denmark41_code]
        },
        {
            "code": time41["id"],
            "values": ["*"]
        }
    ]
)


print("\nRaw Gini data:")
display(gini_raw.head())


# ------------------------------------------------------------
# Identify year and content columns robustly
# ------------------------------------------------------------

print("\nGini columns:")
print(gini_raw.columns.tolist())


# CSV from StatBank normally contains INDHOLD
value_col41 = [
    col for col in gini_raw.columns
    if str(col).upper() == "INDHOLD"
][0]

# Find the year column
year_col41 = [
    col for col in gini_raw.columns
    if "YEAR" in str(col).upper()
    or str(col).upper() == "TID"
][0]


# Clean
gini = gini_raw[
    [year_col41, value_col41]
].copy()

gini.columns = [
    "year",
    "Gini"
]

gini["year"] = pd.to_numeric(
    gini["year"],
    errors="coerce"
)

gini["Gini"] = pd.to_numeric(
    gini["Gini"],
    errors="coerce"
)

gini = (
    gini
    .dropna()
    .sort_values("year")
    .reset_index(drop=True)
)

gini["year"] = gini["year"].astype(int)


print("\nClean Gini data:")
display(gini.head())


# ============================================================
# 2. DECILE DATA FROM IFOR32
# ============================================================

print("\n==============================")
print("IFOR32 - DECILES")
print("==============================")


meta32 = get_metadata("IFOR32")

show_variables(meta32)


# ------------------------------------------------------------
# Detect variables automatically
# ------------------------------------------------------------

decile32 = find_variable(
    meta32,
    ["decile"]
)

municipality32 = find_variable(
    meta32,
    ["municipality", "area"]
)

time32 = find_variable(
    meta32,
    ["year", "time"]
)


print("\nDetected IFOR32 variables:")
print("Decile:", decile32["id"])
print("Municipality:", municipality32["id"])
print("Time:", time32["id"])


# Find All Denmark
denmark32_code = find_value(
    municipality32,
    ["all denmark"]
)


print("\nAll Denmark code:", denmark32_code)


# ------------------------------------------------------------
# Download all 10 deciles
# ------------------------------------------------------------

deciles_raw = download_statbank(
    "IFOR32",
    [
        {
            "code": decile32["id"],
            "values": ["*"]
        },
        {
            "code": municipality32["id"],
            "values": [denmark32_code]
        },
        {
            "code": time32["id"],
            "values": ["*"]
        }
    ]
)


print("\nRaw decile data:")
display(deciles_raw.head())


print("\nDecile columns:")
print(deciles_raw.columns.tolist())


# ============================================================
# 3. IDENTIFY COLUMNS
# ============================================================

value_col32 = [
    col for col in deciles_raw.columns
    if str(col).upper() == "INDHOLD"
][0]


year_col32 = [
    col for col in deciles_raw.columns
    if "YEAR" in str(col).upper()
    or str(col).upper() == "TID"
][0]


# Find decile column by excluding year/value/municipality
candidate_decile_cols = [
    col for col in deciles_raw.columns
    if (
        "DECIL" in str(col).upper()
        or "DECILE" in str(col).upper()
    )
]

if len(candidate_decile_cols) == 0:
    raise ValueError(
        "Could not detect the decile column."
    )

decile_col = candidate_decile_cols[0]


# ============================================================
# 4. CLEAN DECILE DATA
# ============================================================

deciles = deciles_raw[
    [
        year_col32,
        decile_col,
        value_col32
    ]
].copy()


deciles.columns = [
    "year",
    "decile",
    "income"
]


deciles["year"] = pd.to_numeric(
    deciles["year"],
    errors="coerce"
)

deciles["income"] = pd.to_numeric(
    deciles["income"],
    errors="coerce"
)


deciles = deciles.dropna(
    subset=[
        "year",
        "income"
    ]
)

deciles["year"] = deciles["year"].astype(int)


print("\nClean decile data:")
display(deciles.head(15))


# ============================================================
# 5. CHECK THAT WE HAVE 10 DECILES PER YEAR
# ============================================================

deciles_per_year = (
    deciles
    .groupby("year")["decile"]
    .nunique()
)


print("\nNumber of deciles per year:")
print(deciles_per_year.head())


# Only keep years with exactly 10 deciles
valid_years = deciles_per_year[
    deciles_per_year == 10
].index


deciles = deciles[
    deciles["year"].isin(valid_years)
].copy()


# ============================================================
# 6. CALCULATE TOP 10% INCOME SHARE
# ============================================================

# Pivot:
#
# year | First decile | ... | Tenth decile
#

decile_wide = deciles.pivot(
    index="year",
    columns="decile",
    values="income"
)


print("\nDecile table:")
display(decile_wide.head())


print("\nDecile names:")
print(decile_wide.columns.tolist())


# Find tenth decile automatically
top_decile_candidates = [
    col
    for col in decile_wide.columns
    if (
        "tenth" in str(col).lower()
        or "10th" in str(col).lower()
        or str(col).strip() == "10"
    )
]


if len(top_decile_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify the tenth decile.\n"
        f"Columns are: {decile_wide.columns.tolist()}"
    )


top_decile = top_decile_candidates[0]


print("\nDetected top decile:")
print(top_decile)


# ------------------------------------------------------------
# Formula from the assignment:
#
# y_10 / sum(y_1,...,y_10)
#
# Every decile has the same number of people,
# so average incomes can be used directly.
# ------------------------------------------------------------

top10 = pd.DataFrame(
    index=decile_wide.index
)


top10["Top10_share"] = (
    decile_wide[top_decile]
    /
    decile_wide.sum(axis=1)
    *
    100
)


top10 = top10.reset_index()


print("\nTop 10% income share:")
display(top10.head())


# ============================================================
# 7. MERGE THE TWO DATASETS
# ============================================================

data = pd.merge(
    gini,
    top10,
    on="year",
    how="inner",
    validate="1:1"
)


data = (
    data
    .sort_values("year")
    .reset_index(drop=True)
)


print("\n==============================")
print("MERGED DATA")
print("==============================")


display(data.head())

display(data.tail())


print(
    "\nYears used:",
    data["year"].min(),
    "-",
    data["year"].max()
)


# ============================================================
# 8. CORRELATION
# ============================================================

correlation = data[
    "Gini"
].corr(
    data["Top10_share"]
)


print("\n==============================")
print("CORRELATION")
print("==============================")


print(
    f"Correlation between Gini and "
    f"Top 10% income share: "
    f"{correlation:.3f}"
)


# ============================================================
# 9. PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


ax.plot(
    data["year"],
    data["Gini"],
    marker="o",
    markersize=4,
    linewidth=2,
    label="Gini coefficient"
)


ax.plot(
    data["year"],
    data["Top10_share"],
    marker="o",
    markersize=4,
    linewidth=2,
    label="Top 10% income share"
)


ax.set_title(
    "Income Inequality in Denmark"
)


ax.set_xlabel(
    "Year"
)


ax.set_ylabel(
    "Percent"
)


ax.legend()


ax.grid(
    True,
    alpha=0.3
)


plt.tight_layout()

plt.show()


# ============================================================
# 10. SUMMARY STATISTICS
# ============================================================

first = data.iloc[0]
last = data.iloc[-1]


gini_change = (
    last["Gini"]
    -
    first["Gini"]
)


top10_change = (
    last["Top10_share"]
    -
    first["Top10_share"]
)


print("\n==============================")
print("SUMMARY")
print("==============================")


print(
    f"First year: "
    f"{int(first['year'])}"
)


print(
    f"Gini in first year: "
    f"{first['Gini']:.2f}"
)


print(
    f"Top 10% share in first year: "
    f"{first['Top10_share']:.2f}%"
)


print()


print(
    f"Last year: "
    f"{int(last['year'])}"
)


print(
    f"Gini in last year: "
    f"{last['Gini']:.2f}"
)


print(
    f"Top 10% share in last year: "
    f"{last['Top10_share']:.2f}%"
)


print()


print(
    f"Change in Gini: "
    f"{gini_change:+.2f}"
)


print(
    f"Change in Top 10% share: "
    f"{top10_change:+.2f} percentage points"
)


print(
    f"Correlation: "
    f"{correlation:.3f}"
)


# ============================================================
# 11. AUTOMATIC INTERPRETATION
# ============================================================

abs_corr = abs(correlation)


if abs_corr >= 0.8:
    strength = "very strong"

elif abs_corr >= 0.6:
    strength = "strong"

elif abs_corr >= 0.4:
    strength = "moderate"

elif abs_corr >= 0.2:
    strength = "weak"

else:
    strength = "very weak"


if correlation > 0:
    direction = "positive"

elif correlation < 0:
    direction = "negative"

else:
    direction = "zero"


print("\n==============================")
print("INTERPRETATION")
print("==============================")


print(
    f"The correlation between the two measures is "
    f"{correlation:.3f}, indicating a "
    f"{strength} {direction} relationship."
)


if (
    gini_change > 0
    and
    top10_change > 0
):

    print(
        "Both measures indicate that income inequality "
        "has increased over the sample period."
    )


elif (
    gini_change < 0
    and
    top10_change < 0
):

    print(
        "Both measures indicate that income inequality "
        "has decreased over the sample period."
    )


else:

    print(
        "The two measures do not show exactly the same "
        "overall development over the sample period."
    )


print(
    "\nThe Gini coefficient summarizes inequality across "
    "the entire income distribution."
)


print(
    "The Top 10% income share instead measures the share "
    "of total income received by the richest decile."
)


IFOR41 - GINI

Variables in table:
ULLIG  ->  indicator
KOMMUNEDK  ->  municipality
Tid  ->  time

Detected IFOR41 variables:
Indicator: ULLIG
Municipality: KOMMUNEDK
Time: Tid


KeyError: 'valueTexts'